# scikit-learn 1.9: companion notebook

## Dataset

Most examples use the [UCI Bank Marketing dataset](https://archive.ics.uci.edu/dataset/222/bank+marketing). The target, `y`, records whether a customer subscribed to a term deposit. We remove `duration` before training because it is only known after the call ends.


## Setup

Run this cell, then restart the kernel. Run it again before any other cell.


In [ ]:
%pip install -q --upgrade scikit-learn==1.9.1 ucimlrepo array-api-compat


In [ ]:
import sklearn

print(f"scikit-learn version: {sklearn.__version__}")
assert sklearn.__version__.startswith("1.9"), (
    "Restart the kernel, then run the setup cell again."
)


In [ ]:
import importlib

import matplotlib.pyplot as plt
import pandas as pd
from sklearn.callback import ProgressBar, ScoringMonitor
from sklearn.compose import make_column_transformer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    f1_score,
    metric_at_thresholds,
    precision_score,
    recall_score,
)
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
import utils

importlib.reload(utils)
from utils import (
    load_bank_marketing_data,
    load_penguins_data,
    make_customer_spending_data,
)


## Load the data


In [ ]:
data = load_bank_marketing_data()

data.head()

## Check the data


In [ ]:
print(f"Rows: {data.shape[0]:,}")
print(f"Columns: {data.shape[1]}")

data.info()

In [ ]:
missing_values = data.isna().sum().sort_values(ascending=False)
missing_values[missing_values > 0]

In [ ]:
target_counts = data["y"].value_counts()
target_share = data["y"].value_counts(normalize=True).mul(100).round(1)

display(pd.DataFrame({"count": target_counts, "percentage": target_share}))

ax = target_counts.reindex(["no", "yes"]).plot(
    kind="bar", color=["#9ecae1", "#fdae6b"], rot=0, figsize=(6, 4)
)
ax.set_title("Did the customer subscribe to a term deposit?")
ax.set_xlabel("Subscription")
ax.set_ylabel("Customers")
plt.show()

The target is imbalanced. We will use precision, recall, and F1 alongside accuracy.


## Prepare the data

Remove `duration`, then create train, validation, and test splits. The test set is only used for the final comparison.


In [ ]:
X = data.drop(columns=["y", "duration"])
y = (data["y"] == "yes").astype(int)

X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.4, stratify=y, random_state=42
)
X_valid, X_test, y_valid, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, stratify=y_temp, random_state=42
)

numeric_features = X.select_dtypes(include="number").columns
categorical_features = X.select_dtypes(exclude="number").columns

preprocessor = make_column_transformer(
    (StandardScaler(), numeric_features),
    (OneHotEncoder(handle_unknown="ignore"), categorical_features),
)

X_train_prepared = preprocessor.fit_transform(X_train)
X_valid_prepared = preprocessor.transform(X_valid)
X_test_prepared = preprocessor.transform(X_test)

# scikit-learn 1.9 updates


The examples follow a common workflow: data handling, training, inspection, evaluation, visualization, and GPU-backed arrays.


## 1. Missing values in tree models

The examples use a small customer spending dataset with missing feature values.

### Missing values with `absolute_error`

`DecisionTreeRegressor` and `RandomForestRegressor` can now use `criterion="absolute_error"` with missing values in dense data. No imputer is needed.


In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error
from sklearn.tree import DecisionTreeRegressor

customer_data, annual_spending = make_customer_spending_data()

X_tree_train, X_tree_test, y_tree_train, y_tree_test = train_test_split(
    customer_data, annual_spending, test_size=0.25, random_state=42
)

print("Missing values in the full dataset:")
display(customer_data.isna().sum().to_frame("missing values"))

absolute_error_models = {
    "Decision tree": DecisionTreeRegressor(
        criterion="absolute_error", max_depth=5, random_state=42
    ),
    "Random forest": RandomForestRegressor(
        criterion="absolute_error",
        n_estimators=50,
        max_depth=5,
        random_state=42,
    ),
}

absolute_error_scores = {}
for name, model in absolute_error_models.items():
    model.fit(X_tree_train, y_tree_train)
    predictions = model.predict(X_tree_test)
    absolute_error_scores[name] = mean_absolute_error(y_tree_test, predictions)

pd.Series(absolute_error_scores, name="test MAE").sort_values()


The models fit directly on data with missing values. Compare this approach with imputation using validation data.


### Missing values with monotonic constraints

Missing values can also be used with `monotonic_cst`. Here, `[1, -1, 1]` means that predictions increase with income and years as a customer, and decrease with debt ratio.


In [ ]:
monotonic_tree = DecisionTreeRegressor(
    criterion="absolute_error",
    monotonic_cst=[1, -1, 1],
    max_depth=5,
    random_state=42,
)

monotonic_tree.fit(X_tree_train, y_tree_train)
monotonic_predictions = monotonic_tree.predict(X_tree_test)

print(
    "Test MAE:",
    round(mean_absolute_error(y_tree_test, monotonic_predictions), 3),
)


The constrained tree fits without an imputer. This support is for dense `X`; `y` cannot contain missing values.


## 2. Monitor training with callbacks

`ProgressBar` shows fit progress. `ScoringMonitor` records metrics during fitting. This example monitors `neg_log_loss` and `average_precision`.

For `LogisticRegression`, callbacks require `solver="lbfgs"`. The callback API is experimental.


In [ ]:
progress_bar = ProgressBar()
scoring_monitor = ScoringMonitor(scoring=["neg_log_loss", "average_precision"])

logreg = LogisticRegression(solver="lbfgs", max_iter=1000, random_state=42)
logreg.set_callbacks(progress_bar, scoring_monitor)
logreg.fit(X_train_prepared, y_train)

Inspect the metrics recorded during fitting.


In [ ]:
log = scoring_monitor.get_logs().data_as_pandas
log[["task_name", "task_id", "neg_log_loss", "average_precision"]].head()

The same progress bar can also be attached to a grid search.


In [ ]:
search_pipeline = make_pipeline(
    preprocessor,
    LogisticRegression(solver="lbfgs", max_iter=1000, random_state=42),
)

grid_search = GridSearchCV(
    search_pipeline,
    param_grid={"logisticregression__C": [0.01, 0.1, 1, 10]},
    cv=3,
    n_jobs=2,
)
grid_search.set_callbacks(ProgressBar())
grid_search.fit(X_train, y_train)

print("Best parameters:", grid_search.best_params_)

## 3. Inspect fitted estimators

After fitting, the notebook display includes a **Fitted attributes** table. Transformers also show their output features.


In [ ]:
from sklearn import set_config

set_config(display="diagram")

inspection_pipeline = make_pipeline(
    preprocessor,
    LogisticRegression(solver="lbfgs", max_iter=1000, random_state=42),
)
fitted_pipeline = inspection_pipeline.fit(X_train, y_train)

display(fitted_pipeline)
display(fitted_pipeline.named_steps["columntransformer"])
display(fitted_pipeline.named_steps["logisticregression"])

## 4. Inspect metrics across decision thresholds

`metric_at_thresholds` calculates a metric at every threshold where predictions can change. We use the validation set to calculate and plot accuracy, precision, and recall.


In [ ]:
probabilities = logreg.predict_proba(X_valid_prepared)[:, 1]


First, view the calculated values. Then plot the same values.


In [ ]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    metric_at_thresholds,
)

accuracy, thresholds = metric_at_thresholds(y_valid, probabilities, accuracy_score)
precision, _ = metric_at_thresholds(y_valid, probabilities, precision_score)
recall, _ = metric_at_thresholds(y_valid, probabilities, recall_score)

metrics_by_threshold = pd.DataFrame(
    {
        "threshold": thresholds,
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
    }
).sort_values("threshold", ignore_index=True)

metrics_by_threshold


In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(
    metrics_by_threshold["threshold"],
    metrics_by_threshold["accuracy"],
    label="Accuracy",
)
plt.plot(
    metrics_by_threshold["threshold"],
    metrics_by_threshold["precision"],
    label="Precision",
)
plt.plot(
    metrics_by_threshold["threshold"],
    metrics_by_threshold["recall"],
    label="Recall",
)
plt.xlabel("Decision threshold")
plt.ylabel("Metric value")
plt.title("How metrics change with the decision threshold")
plt.legend()
plt.show()


Compare the default threshold with the threshold that gives the highest validation accuracy. Use the test set only for this final comparison.


In [ ]:
best_threshold = thresholds[accuracy.argmax()]
test_probabilities = logreg.predict_proba(X_test_prepared)[:, 1]

default_predictions = (test_probabilities >= 0.5).astype(int)
selected_predictions = (test_probabilities >= best_threshold).astype(int)

comparison = pd.DataFrame(
    {
        "threshold": [0.5, best_threshold],
        "precision": [
            precision_score(y_test, default_predictions),
            precision_score(y_test, selected_predictions),
        ],
        "recall": [
            recall_score(y_test, default_predictions),
            recall_score(y_test, selected_predictions),
        ],
        "f1": [
            f1_score(y_test, default_predictions),
            f1_score(y_test, selected_predictions),
        ],
    },
    index=["Default threshold", "Best validation accuracy"],
)
comparison.round(3)

Choose a threshold based on the cost of errors in the use case.


## 5. More accessible multiclass decision boundaries

The Palmer Penguins dataset has three species. We use bill length and bill depth to plot a multiclass decision boundary.

In version 1.9, `DecisionBoundaryDisplay` uses the accessible Petroff colour sequence by default for multiclass plots with up to ten classes. We use the same colours for the points and decision regions.


In [ ]:
import importlib
import utils

importlib.reload(utils)
from utils import load_penguins_data

import matplotlib.pyplot as plt
import pandas as pd
import sklearn
from matplotlib.colors import ListedColormap
from sklearn.inspection import DecisionBoundaryDisplay
from sklearn.linear_model import LogisticRegression

assert sklearn.__version__.startswith("1.9")

penguins = load_penguins_data()
plot_data = penguins[["bill_length_mm", "bill_depth_mm", "species"]].dropna()

X_penguins = plot_data[["bill_length_mm", "bill_depth_mm"]]
y_penguins = pd.Categorical(plot_data["species"]).codes

model = LogisticRegression(max_iter=300).fit(X_penguins, y_penguins)

disp = DecisionBoundaryDisplay.from_estimator(
    model,
    X_penguins,
    response_method="predict",
    alpha=0.5,
)

disp.ax_.scatter(
    X_penguins.iloc[:, 0],
    X_penguins.iloc[:, 1],
    c=y_penguins,
    cmap=ListedColormap(disp.multiclass_colors_),
    edgecolor="black",
)
plt.show()


## 6. More Array API support

Array API dispatch lets supported scikit-learn operations work directly with compatible arrays, such as PyTorch tensors on a GPU. The feature is experimental.

Run the next cell, restart the kernel, then run this section. `SCIPY_ARRAY_API` must be set before scikit-learn is imported.

Here, `LogisticRegression` with `solver="lbfgs"` fits directly on CUDA tensors. `cuda:0` confirms that the predictions and learned coefficients remain on the first GPU.


In [ ]:
import os

# Enable Array API support before importing scikit-learn
os.environ["SCIPY_ARRAY_API"] = "1"


In [ ]:
import torch
from sklearn import config_context
from sklearn.datasets import make_classification
from sklearn.linear_model import LogisticRegression

if not torch.cuda.is_available():
    raise RuntimeError("This example requires a CUDA GPU.")

X_np, y_np = make_classification(
    n_samples=1_000,
    n_features=20,
    n_informative=10,
    random_state=42,
)

X_torch = torch.asarray(X_np, dtype=torch.float32, device="cuda")
y_torch = torch.asarray(y_np, dtype=torch.int64, device="cuda")

with config_context(array_api_dispatch=True):
    model = LogisticRegression(solver="lbfgs", max_iter=500).fit(
        X_torch, y_torch
    )
    predictions = model.predict(X_torch)

print("Prediction device:", predictions.device)
print("Coefficient device:", model.coef_.device)


## Further reading

See the [scikit-learn 1.9 release notes](https://scikit-learn.org/1.9/whats_new/v1.9.html) for all changes and supported estimators.
